# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [ ]:
%help

####  Run this cell to set up and start your interactive session.


In [5]:
%idle_timeout 2880
%glue_version 5.0
%worker_type G.1X
%number_of_workers 3

In [1]:
import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql.functions import to_timestamp, col, regexp_replace
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.types import (
    StringType, IntegerType, DoubleType, DateType, 
    TimestampType, BooleanType, DecimalType, StructType
)
import re

In [3]:
spark = SparkSession.builder \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.glue_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.glue_catalog.warehouse", "s3://prism-nih-bronze/") \
    .config("spark.sql.catalog.glue_catalog.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog") \
    .config("spark.sql.catalog.glue_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .getOrCreate()

sc = spark.sparkContext
glueContext = GlueContext(sc)
job = Job(glueContext)

In [4]:
df = spark.table("glue_catalog.prism_bronze.demo_nmrr_sites_description")



In [5]:
df.show(5)

In [6]:
 # Execute SQL directly against the Glue catalog
spark.sql("""
    SELECT * FROM glue_catalog.prism_bronze.demo_nmrr_sites_description 
""").show()

In [7]:
df = df.withColumn(
    "ingestion_date",
    col("ingestion_date_str").cast("date")
)

In [8]:
df.printSchema()

In [28]:
#df = df.withColumnRenamed("nmrr_id", "nmrr_id_sites").withColumnRenamed("pi_affiliation", "pi_affiliation_sites").withColumnRenamed("research_id", "research_id_sites")

In [9]:
df.writeTo("glue_catalog.prism_silver.demo_nmrr_sites_description_trnst").tableProperty("format-version", "2").tableProperty("location", "s3://prism-nih-silver/prism_silver/demo_nmrr_sites_description_trnst").createOrReplace()
    

In [16]:
# import boto3

# glue_client = boto3.client('glue')

# response = glue_client.delete_table(
#     DatabaseName='prism_silver',
#     Name='demo_nmrr_sites_description_trn'
# )
# print("Ghost table successfully deleted from Glue.")

In [4]:
nmrr_df = spark.table("glue_catalog.prism_bronze.demo_nmrr_general_information")

In [11]:
nmrr_df = nmrr_df.withColumn("cleaned_id", regexp_replace(col("nmrr_id"), " ID", ""))

In [13]:
nmrr_df.select(col("cleaned_id")).show(5)

In [29]:
joined_table = nmrr_df.join(df, df.nmrr_id_sites == nmrr_df.cleaned_id, how="left")

In [30]:
joined_table.show(5)

In [ ]:
def generate_merge_query(df: DataFrame, table_name: str, **kwargs) -> str:
    """
    Generates a SQL MERGE INTO statement from a Spark DataFrame.

    This function dynamically constructs a MERGE query to upsert data from a
    source (represented by a temporary view of the DataFrame) into a target table.

    :param df: The source Spark DataFrame containing new data.
    :param table_name: The name of the target table to merge into.
    :param kwargs: Keyword arguments for controlling the merge logic.
        - on_keys (list[str]): A list of column names to use for the join
          condition. This is a mandatory argument.
        - source_view (str): The name for the temporary view to be created from
          the DataFrame. Defaults to 'source_view'.
        - target_alias (str): The SQL alias for the target table. Defaults to 'T'.
        - source_alias (str): The SQL alias for the source view. Defaults to 'S'.
    :return: A formatted SQL MERGE INTO statement as a string.
    :raises ValueError: If 'on_keys' is not provided or is empty.
    """
    # --- 1. Validate and Extract Parameters ---
    on_keys = kwargs.get('on_keys')
    if not on_keys:
        raise ValueError("'on_keys' is a mandatory keyword argument and cannot be empty.")

    source_view = kwargs.get('source_view', f'v_{table_name}')
    target_alias = kwargs.get('target_alias', 'T')
    source_alias = kwargs.get('source_alias', 'S')

    all_columns = df.columns
    update_columns = [col for col in all_columns if col not in on_keys]

    # --- 2. Construct Query Clauses ---

    # ON clause for joining source and target
    # Example: T.Encounter_ID = S.Encounter_ID AND T.Patient_ID = S.Patient_ID
    on_clause = " AND ".join([f"{target_alias}.{key} = {source_alias}.{key}" for key in on_keys])

    # WHEN MATCHED clause to update existing records
    # Example: T.Diagnosis_Code = S.Diagnosis_Code, T.Billing_Amount = S.Billing_Amount
    update_clause = ",\n    ".join([f"{target_alias}.{col} = {source_alias}.{col}" for col in update_columns])
    # WHEN NOT MATCHED clause to insert new records
    # Example: (Encounter_ID, Patient_ID) VALUES (S.Encounter_ID, S.Patient_ID)
    insert_columns = ",\n    ".join([f"{col}" for col in all_columns])
    insert_values = ",\n    ".join([f"{source_alias}.{col}" for col in all_columns])

    # --- 3. Assemble the Final MERGE Statement ---
    merge_sql = f"""
    MERGE INTO {table_name} AS {target_alias}
    USING {source_view} AS {source_alias}
    ON {on_clause}
    WHEN MATCHED THEN
      UPDATE SET
        {update_clause}
    WHEN NOT MATCHED THEN
      INSERT (
        {insert_columns}
      ) VALUES (
        {insert_values}
      )
    """
    return merge_sql

In [ ]:
view_name = "nmrr_temp_view"
spark_df.createOrReplaceTempView(view_name)

In [31]:
joined_table.writeTo("glue_catalog.prism_silver.demo_nmrr_general_information_join").tableProperty("format-version", "2").tableProperty("location", "s3://prism-nih-silver/prism_silver/demo_nmrr_general_information_join").createOrReplace()